# 04 - Modelo predictivo

Reentrena un modelo simple (Random Forest) en el notebook para explorar el enfoque de `src/train_model.py` de forma interactiva: target = cambio semestral de TD (no el nivel), split temporal train/test, y comparación contra un baseline ingenuo (random walk).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from config import INTERMEDIATE_DIR, PRIMARY_DIR, MODEL_OUTPUT_DIR

pd.set_option("display.max_columns", 50)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sys.path.insert(0, str(REPO_ROOT / "src"))
from train_model import FEATURES, TARGET_LEVEL, TEST_CUTOFF_YEAR

panel = pd.read_csv(PRIMARY_DIR / "panel_final.csv")
modelable = panel.dropna(subset=FEATURES + [TARGET_LEVEL, "td"]).copy()
modelable["td_delta"] = modelable[TARGET_LEVEL] - modelable["td"]

train = modelable[modelable["anio"] < TEST_CUTOFF_YEAR]
test = modelable[modelable["anio"] >= TEST_CUTOFF_YEAR]
len(train), len(test)

In [ ]:
model = RandomForestRegressor(n_estimators=400, max_depth=4, min_samples_leaf=4, random_state=42)
model.fit(train[FEATURES], train["td_delta"])

pred_delta = model.predict(test[FEATURES])
pred_level = test["td"].values + pred_delta

naive_level = test["td"].values  # baseline: TD no cambia

print("MAE modelo :", mean_absolute_error(test[TARGET_LEVEL], pred_level))
print("MAE naive  :", mean_absolute_error(test[TARGET_LEVEL], naive_level))
print("R2 modelo  :", r2_score(test[TARGET_LEVEL], pred_level))

## Real vs. predicho

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(test[TARGET_LEVEL], pred_level, alpha=0.7)
lims = [test[TARGET_LEVEL].min(), test[TARGET_LEVEL].max()]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("TD real (%)"); ax.set_ylabel("TD predicho (%)")
ax.set_title("Random Forest (notebook) — real vs. predicho")
plt.show()

## Importancia de variables

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.head(10)

Nota: `src/train_model.py` compara 5 modelos (LinearRegression, Ridge, Random Forest, Gradient Boosting, XGBoost) con validación cruzada temporal (`TimeSeriesSplit`) y guarda el mejor en `models/`; aquí solo se explora uno para mantener el notebook simple.